In [ ]:
# Install required libraries and fix torchvision/VideoReader dependency
!pip -q install pandas numpy scikit-learn transformers datasets accelerate torch seaborn matplotlib tqdm

In [ ]:
# Imports and reproducibility
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import warnings
from scipy.special import softmax
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set(style='whitegrid')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


In [ ]:
# Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/multisocial_outputs'
TRAIN_PATH = os.path.join(BASE_DIR, 'multisocial_train.csv')
TEST_PATH = os.path.join(BASE_DIR, 'multisocial_test.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'finetuned_xlmr')
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
RESULTS_JSON = os.path.join(OUTPUT_DIR, 'results_finetuned_per_language.json')

MODEL_NAME = 'xlm-roberta-base'
MAX_LENGTH = 256
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
TRAIN_BS = 16
EVAL_BS = 16

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# CSV output directories
RESULTS_DIR = '/content/results'
DRIVE_RESULTS_DIR = OUTPUT_DIR
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(TRAIN_PATH), f'Train file not found: {TRAIN_PATH}'
assert os.path.exists(TEST_PATH), f'Test file not found: {TEST_PATH}'
print(f'Train path: {TRAIN_PATH}')
print(f'Test path: {TEST_PATH}')
print(f'Output dir: {OUTPUT_DIR}')

Mounted at /content/drive
Train path: /content/drive/MyDrive/multisocial_outputs/multisocial_train.csv
Test path: /content/drive/MyDrive/multisocial_outputs/multisocial_test.csv
Output dir: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr


In [ ]:
# Load and validate train/test datasets
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

required_cols = {'text', 'label', 'language'}
for name, df in [('train', train_df), ('test', test_df)]:
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{name} dataframe missing columns: {sorted(missing)}')

    df.dropna(subset=['text', 'label', 'language'], inplace=True)
    df['text'] = df['text'].astype(str)
    df['label'] = df['label'].astype(int)
    df['language'] = df['language'].astype(str)

assert not train_df.empty, 'Train dataframe is empty after cleanup.'
assert not test_df.empty, 'Test dataframe is empty after cleanup.'

print(f'Train rows: {len(train_df)}')
print(f'Test rows: {len(test_df)}')
print('Test rows by language:')
print(test_df['language'].value_counts())

Train rows: 12789
Test rows: 3197
Test rows by language:
language
en    800
zh    800
ar    800
vi    797
Name: count, dtype: int64


In [ ]:
# Build DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True)),
})

assert len(dataset['train']) > 0, 'HF train dataset is empty.'
assert len(dataset['test']) > 0, 'HF test dataset is empty.'
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 12789
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3197
    })
})


In [ ]:
# Tokenizer, tokenization, and data collator
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

tokenized = dataset.map(tokenize_batch, batched=True)
tokenized = tokenized.remove_columns(['text'])
tokenized.set_format(type='torch')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

assert 'input_ids' in tokenized['train'].features, 'Tokenization failed: input_ids missing.'
print('Tokenization complete.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/12789 [00:00<?, ? examples/s]

Map:   0%|          | 0/3197 [00:00<?, ? examples/s]

Tokenization complete.


In [ ]:
# Trainer metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=-1)[:, 1]  # machine class probability
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    if len(np.unique(labels)) < 2:
        auc = float('nan')
    else:
        auc = float(roc_auc_score(labels, probs))
    return {'accuracy': float(acc), 'macro_f1': float(macro_f1), 'roc_auc': auc}

In [ ]:
# Model and TrainingArguments
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to=[],
    seed=SEED,
)

print('Training config prepared.')
print(f'Batch size train/eval: {TRAIN_BS}/{EVAL_BS}')
print('If OOM occurs, re-run with TRAIN_BS=8, EVAL_BS=8, gradient_accumulation_steps=2.')

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training config prepared.
Batch size train/eval: 16/16
If OOM occurs, re-run with TRAIN_BS=8, EVAL_BS=8, gradient_accumulation_steps=2.


In [ ]:
import sys
import datasets.config

# Explicitly disable torchvision check in datasets to prevent VideoReader ImportError
datasets.config.TORCHVISION_AVAILABLE = False

# Train with OOM-aware fallback
try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['test'],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    train_output = trainer.train()
except torch.cuda.OutOfMemoryError:
    print('OOM encountered. Retrying with smaller batch and gradient accumulation...')
    torch.cuda.empty_cache()

    fallback_args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to=[],
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=fallback_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['test'],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    train_output = trainer.train()

print('Training complete.')

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Roc Auc
1,0.431197,0.432371,0.788865,0.787459,0.878369
2,0.359918,0.424019,0.799187,0.799002,0.904767
3,0.320205,0.408793,0.813575,0.813538,0.914365


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete.


In [ ]:
# Evaluate and get predictions
eval_metrics = trainer.evaluate()
pred_output = trainer.predict(tokenized['test'])

print('Evaluation metrics from Trainer:')
print(eval_metrics)

y_true = pred_output.label_ids.astype(int)
y_pred = np.argmax(pred_output.predictions, axis=-1).astype(int)
y_prob = softmax(pred_output.predictions, axis=-1)[:, 1].astype(np.float32)  # machine class

assert len(y_true) == len(test_df), 'Prediction count does not match test dataframe.'
assert len(y_pred) == len(test_df), 'Predicted label count does not match test dataframe.'
assert len(y_prob) == len(test_df), 'Probability count does not match test dataframe.'

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Roc Auc
0.320205,0.408793,3,0.813575,0.813538,0.914365


Evaluation metrics from Trainer:
{'eval_loss': 0.4087926149368286, 'eval_accuracy': 0.8135752267751016, 'eval_macro_f1': 0.8135382840071774, 'eval_roc_auc': 0.9143648117183636}


In [ ]:
# Build per-language metrics and overall metrics
eval_df = test_df.copy().reset_index(drop=True)
eval_df['pred_label'] = y_pred
eval_df['prob_machine'] = y_prob

# Helper functions
def safe_roc_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        warnings.warn(f'Only one class in y_true, ROC-AUC undefined. Returning NaN.')
        return float('nan')
    return float(roc_auc_score(y_true, y_score))

def compute_classification_metrics(y_true, y_prob, threshold=0.5):
    y_pred_local = (y_prob >= threshold).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred_local)),
        'precision_macro': float(precision_score(y_true, y_pred_local, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_true, y_pred_local, average='macro', zero_division=0)),
        'f1_macro': float(f1_score(y_true, y_pred_local, average='macro', zero_division=0)),
        'roc_auc': safe_roc_auc(y_true, y_prob),
    }

def save_df(df, path, index=False):
    df.to_csv(path, index=index)
    print(f'Saved: {path} ({len(df)} rows)')

# Per-language metrics
lang_results = []
for lang in ['en', 'vi', 'zh', 'ar']:
    lang_df = eval_df[eval_df['language'] == lang].copy()
    assert not lang_df.empty, f'No rows in test data for language {lang}'

    metrics = compute_classification_metrics(
        lang_df['label'].values.astype(int),
        lang_df['prob_machine'].values.astype(np.float32)
    )
    metrics['language'] = lang
    metrics['n_samples'] = int(len(lang_df))
    lang_results.append(metrics)

lang_results_df = pd.DataFrame(lang_results)
lang_results_df = lang_results_df[['language', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc', 'n_samples']]

# Overall metrics
overall_metrics = compute_classification_metrics(
    eval_df['label'].values.astype(int),
    eval_df['prob_machine'].values.astype(np.float32)
)
overall_metrics['language'] = 'overall'
overall_metrics['n_samples'] = int(len(eval_df))
overall_df = pd.DataFrame([overall_metrics])
overall_df = overall_df[['language', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc', 'n_samples']]

# Display combined summary
full_results_df = pd.concat([overall_df, lang_results_df], ignore_index=True)
print('Fine-tuned XLM-RoBERTa evaluation metrics:')
display(full_results_df)

assert len(lang_results_df) == 4, 'Expected results for 4 languages.'

Fine-tuned XLM-RoBERTa evaluation metrics:


,language,accuracy,precision_macro,recall_macro,f1_macro,roc_auc,n_samples
0,overall,0.813575,0.814270,0.813827,0.813538,0.914365,3197
1,en,0.953750,0.953753,0.953750,0.953750,0.984234,800
2,vi,0.702635,0.702138,0.701343,0.701552,0.781657,797
3,zh,0.715000,0.723499,0.715000,0.712265,0.811128,800
4,ar,0.882500,0.883113,0.882500,0.882453,0.962028,800


In [ ]:
# Save CSV exports to local and Drive
# 1. Overall metrics
save_df(overall_df, os.path.join(RESULTS_DIR, 'finetuned_overall_metrics.csv'))
save_df(overall_df, os.path.join(DRIVE_RESULTS_DIR, 'finetuned_overall_metrics.csv'))

# 2. Per-language metrics
save_df(lang_results_df, os.path.join(RESULTS_DIR, 'finetuned_per_language_metrics.csv'))
save_df(lang_results_df, os.path.join(DRIVE_RESULTS_DIR, 'finetuned_per_language_metrics.csv'))

# 3. Full test predictions
pred_df = eval_df[['text', 'label', 'language', 'prob_machine', 'pred_label']].copy()
save_df(pred_df, os.path.join(RESULTS_DIR, 'finetuned_test_predictions.csv'))
save_df(pred_df, os.path.join(DRIVE_RESULTS_DIR, 'finetuned_test_predictions.csv'))

print('All CSV exports complete.')

Saved: /content/results/finetuned_overall_metrics.csv (1 rows)
Saved: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/finetuned_overall_metrics.csv (1 rows)
Saved: /content/results/finetuned_per_language_metrics.csv (4 rows)
Saved: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/finetuned_per_language_metrics.csv (4 rows)
Saved: /content/results/finetuned_test_predictions.csv (3197 rows)
Saved: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/finetuned_test_predictions.csv (3197 rows)
All CSV exports complete.


In [ ]:
# Save confusion matrices per language
cm_paths = {}
for lang in ['en', 'vi', 'zh', 'ar']:
    lang_df = eval_df[eval_df['language'] == lang].copy()
    cm = confusion_matrix(lang_df['label'].values, lang_df['pred_label'].values, labels=[0, 1])

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['human (0)', 'machine (1)'],
        yticklabels=['human (0)', 'machine (1)'],
    )
    plt.title(f'Confusion Matrix - {lang}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()

    cm_path = os.path.join(OUTPUT_DIR, f'cm_{lang}.png')
    plt.savefig(cm_path, dpi=200)
    plt.close()

    cm_paths[lang] = cm_path
    assert os.path.exists(cm_path), f'Confusion matrix not saved: {cm_path}'

print('Saved confusion matrices:')
for lang, path in cm_paths.items():
    print(f'  {lang}: {path}')

Saved confusion matrices:
  en: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/cm_en.png
  vi: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/cm_vi.png
  zh: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/cm_zh.png
  ar: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/cm_ar.png


In [ ]:
# Plot combined ROC curve per language
from sklearn.metrics import roc_curve

roc_plot_path = os.path.join(OUTPUT_DIR, 'roc_combined.png')

plt.figure(figsize=(7, 6))
for lang in ['en', 'vi', 'zh', 'ar']:
    lang_df = eval_df[eval_df['language'] == lang].copy()
    y_true_lang = lang_df['label'].values.astype(int)
    y_prob_lang = lang_df['prob_machine'].values.astype(np.float32)
    auc_val = safe_roc_auc(y_true_lang, y_prob_lang)
    fpr, tpr, _ = roc_curve(y_true_lang, y_prob_lang)
    plt.plot(fpr, tpr, label=f'{lang} (AUC={auc_val:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.7)
plt.title(f'Combined ROC Curves — Fine-tuned {MODEL_NAME}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(roc_plot_path, dpi=220)
plt.close()

assert os.path.exists(roc_plot_path), f'ROC plot not saved: {roc_plot_path}'
print(f'Saved ROC plot: {roc_plot_path}')

Saved ROC plot: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/roc_combined.png


In [ ]:
# Save final metrics and checkpoint checks
best_ckpt = trainer.state.best_model_checkpoint
if best_ckpt is None:
    # Fallback to output dir when best checkpoint is unavailable
    best_ckpt = CHECKPOINT_DIR

assert os.path.exists(best_ckpt), f'Best/fallback checkpoint path does not exist: {best_ckpt}'

payload = {
    'seed': SEED,
    'model_name': MODEL_NAME,
    'train_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
    'trainer_eval': {k: float(v) for k, v in eval_metrics.items() if isinstance(v, (int, float))},
    'overall': overall_df.to_dict(orient='records')[0],
    'per_language': lang_results_df.to_dict(orient='records'),
    'best_checkpoint': best_ckpt,
    'confusion_matrices': cm_paths,
    'roc_plot_path': roc_plot_path,
}

with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

assert os.path.exists(RESULTS_JSON), f'Results JSON not found after save: {RESULTS_JSON}'
print(f'Saved fine-tuning results to: {RESULTS_JSON}')
print(f'Best checkpoint: {best_ckpt}')

Saved fine-tuning results to: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/results_finetuned_per_language.json
Best checkpoint: /content/drive/MyDrive/multisocial_outputs/finetuned_xlmr/checkpoints/checkpoint-2400


In [ ]:
# Optional cleanup to free memory before next notebook
del pred_output
del eval_df
del model
del tokenizer
del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Cleanup complete. Done.')

Cleanup complete. Done.
